# Work with binary files

You need to read or write non-text data such as images, audio files, or custom binary formats. Text mode will corrupt binary data because it performs encoding conversion and newline translation.

This guide covers how to work with binary files safely and efficiently.

## Understanding text mode versus binary mode

Python offers two modes for opening files:

- **Text mode** (`"r"`, `"w"`) -- works with `str`, decodes and encodes bytes, translates newlines
- **Binary mode** (`"rb"`, `"wb"`) -- works with `bytes`, no decoding, no newline translation

In [ ]:
from pathlib import Path

Path("test.txt").write_text("Hello\nWorld", encoding="utf-8")

with open("test.txt", "r", encoding="utf-8") as f:
    text_data = f.read()
    print(f"Text mode: type={type(text_data).__name__}, value={text_data!r}")

with open("test.txt", "rb") as f:
    binary_data = f.read()
    print(f"Binary mode: type={type(binary_data).__name__}, value={binary_data!r}")

## Reading binary files

Use `"rb"` mode to read binary data. The result is a `bytes` object.

In [ ]:
from pathlib import Path

sample_bytes = bytes(range(256))
Path("sample.bin").write_bytes(sample_bytes)

with open("sample.bin", "rb") as f:
    data = f.read()

print(f"Read {len(data)} bytes")
print(f"First 10 bytes: {data[:10]}")
print(f"As hex: {data[:10].hex()}")

## Writing binary files

Use `"wb"` mode to write binary data. Do not specify the `encoding` parameter -- binary mode works directly with bytes.

In [ ]:
from pathlib import Path

header = b"\x89PNG\r\n\x1a\n"
data = b"\x00" * 100

with open("sample-data.bin", "wb") as f:
    f.write(header)
    f.write(data)

file_size = Path("sample-data.bin").stat().st_size
print(f"Wrote {file_size} bytes")

## Copying binary files

A practical pattern is copying files in binary mode using chunks. This works for any file type and handles large files efficiently.

In [ ]:
from pathlib import Path


def copy_file(
    source: str | Path, destination: str | Path, chunk_size: int = 8192
) -> int:
    """Copy a file in binary mode using chunks.

    Args:
        source: The path to the source file.
        destination: The path to the destination file.
        chunk_size: The number of bytes to read at a time.

    Returns:
        The total number of bytes copied.
    """
    source_path = Path(source)
    dest_path = Path(destination)
    total = 0
    with source_path.open("rb") as src, dest_path.open("wb") as dst:
        while True:
            chunk = src.read(chunk_size)
            if not chunk:
                break
            dst.write(chunk)
            total += len(chunk)
    return total


bytes_copied = copy_file("sample.bin", "sample-copy.bin")
print(f"Copied {bytes_copied} bytes")

original = Path("sample.bin").read_bytes()
copy = Path("sample-copy.bin").read_bytes()
print(f"Files are identical: {original == copy}")

## Using `pathlib` for simple binary operations

`Path.read_bytes()` and `Path.write_bytes()` provide a convenient way to read and write binary data in one step.

In [ ]:
from pathlib import Path

data = b"Binary content: \x00\x01\x02\x03"
Path("quick.bin").write_bytes(data)

loaded = Path("quick.bin").read_bytes()
print(f"Content: {loaded}")
print(f"Match: {data == loaded}")

## Working with the `struct` module

The `struct` module packs and unpacks structured binary data. This is useful for reading and writing binary file formats with defined layouts.

In [ ]:
import struct

# Pack: unsigned int, unsigned short, float (big-endian)
record = struct.pack(">I H f", 12345, 42, 3.14)
print(f"Packed: {record.hex()}")
print(f"Size: {len(record)} bytes")

with open("record.bin", "wb") as f:
    f.write(record)

with open("record.bin", "rb") as f:
    data = f.read()
    unpacked = struct.unpack(">I H f", data)

print(f"Unpacked: id={unpacked[0]}, count={unpacked[1]}, value={unpacked[2]:.2f}")

The format string `">I H f"` means:

- `>` -- big-endian byte order
- `I` -- unsigned 32-bit integer
- `H` -- unsigned 16-bit integer
- `f` -- 32-bit floating point

## Key takeaways

- Use `"rb"` and `"wb"` for binary files
- Do not specify `encoding` in binary mode
- Binary operations work with `bytes`, not `str`
- Use `Path.read_bytes()` and `Path.write_bytes()` for simple operations
- Use the `struct` module for structured binary data
- Process large binary files in chunks for memory efficiency

In [ ]:
from pathlib import Path

for filename in ["test.txt", "sample.bin", "sample-data.bin",
                 "sample-copy.bin", "quick.bin", "record.bin"]:
    Path(filename).unlink(missing_ok=True)

print("Temporary files removed.")